# Laya-V vs general VLMs: Colab T4 benchmark

Runs Laya-V on this T4 (latency + accuracy) and 6 VLMs through OpenRouter, on the same questions and images.

**Before you run**
1. *Runtime → Change runtime type →* **T4 GPU**.
2. Put `laya-v-code.tar.gz` in Google Drive at `MyDrive/laya-v-bench/`.
3. Add two Colab secrets (key icon in the left bar), both with notebook access on: `HF_TOKEN` (read access to
   `henilchopada/laya-v` and `henilchopada/laya-v-data`) and `OPENROUTER_API_KEY`.
4. *Runtime → Run all.* Setup takes about 15–25 min, latency about 5 min, the benchmark about 30–60 min.

Everything the benchmark writes goes to `MyDrive/laya-v-bench/runs/`. If the runtime disconnects, run all again:
API answers are cached there, so nothing is paid twice. At the end, send back
`MyDrive/laya-v-bench/laya-v-bench-results.zip` (also downloaded by the last cell).

In [ ]:
# ---- settings
USD_PER_HOUR = 0.35   # reference T4 on-demand price (about GCP's), for Laya-V's $ per 1,000 decisions
MAX_USD = 30          # OpenRouter spend cap per model
PER_TASK = 300        # rows per task; 300 = the published eval subset (4,872 rows)
MODELS = []           # empty = the default six (teacher, Sonnet 5, GPT-6 Sol, Gemini 3.8 Flash, Qwen3-VL-8B, GPT-6 Luna)
DRIVE_DIR = "/content/drive/MyDrive/laya-v-bench"

In [ ]:
import os, subprocess
from google.colab import drive, userdata

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout
print("GPU:", gpu.strip() or "none")
assert "T4" in gpu, "Switch the runtime to a T4 GPU first"
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR + "/runs/logs", exist_ok=True)
for k in ("HF_TOKEN", "OPENROUTER_API_KEY"):
    os.environ[k] = userdata.get(k)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
TAR = DRIVE_DIR + "/laya-v-code.tar.gz"
assert os.path.exists(TAR), "Put laya-v-code.tar.gz in " + DRIVE_DIR

## 1. Code and packages

In [ ]:
!rm -rf /content/laya && mkdir -p /content/laya && tar -xzf "$TAR" -C /content/laya
!pip install -q -U "transformers>=5.0" safetensors huggingface_hub pyarrow pillow httpx einops matplotlib
!pip install -q -e /content/laya --no-deps
!python -c "import torch, transformers; print('torch', torch.__version__, 'transformers', transformers.__version__)"

## 2. Checkpoint, items and eval images

In [ ]:
import json, glob
from huggingface_hub import snapshot_download

D, RAW, CK = "/content/data", "/content/raw", "/content/ckpt/laya-v"
os.makedirs(D + "/items", exist_ok=True)
snapshot_download("henilchopada/laya-v", local_dir=CK)
snapshot_download("google/siglip2-so400m-patch16-naflex")
snapshot_download("henilchopada/laya-v-data", repo_type="dataset", local_dir=D,
                  allow_patterns=["built_items/stage_b.jsonl", "built_items/audit_items.jsonl",
                                  "built_items/audit_verdicts_sonnet5.json", "built_items/captions_coco.jsonl"])
for f in glob.glob(D + "/built_items/*"):
    os.replace(f, D + "/items/" + os.path.basename(f))

# teacher-pool eval images: only the ones eval / audit rows use (items say pool/images/..., the Hub has images/...)
need = set()
for p in ("stage_b.jsonl", "audit_items.jsonl"):
    for line in open(D + "/items/" + p, encoding="utf-8"):
        r = json.loads(line)
        if r.get("image") and (r.get("split") == "eval" or p.startswith("audit")):
            need.add(r["image"])
pool = sorted(p[len("pool/"):] for p in need if p.startswith("pool/images/"))
print("eval images", len(need), "from the teacher pool", len(pool))
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
todo = [p for p in pool if not os.path.exists(os.path.join(D, p))]
with ThreadPoolExecutor(16) as ex:  # exact paths: some file names contain [ ], which glob patterns would mangle
    list(ex.map(lambda p: hf_hub_download("henilchopada/laya-v-data", p, repo_type="dataset", local_dir=D), todo))
os.makedirs(D + "/pool", exist_ok=True)
if not os.path.exists(D + "/pool/images"):
    os.symlink(D + "/images", D + "/pool/images")

# public eval splits only
snapshot_download("HuggingFaceM4/A-OKVQA", repo_type="dataset", local_dir=RAW + "/aokvqa", allow_patterns=["*validation-*"])
snapshot_download("naver-clova-ix/cord-v2", repo_type="dataset", local_dir=RAW + "/cord",
                  allow_patterns=["data/validation-*", "data/test-*"])
snapshot_download("rootsautomation/ScreenSpot", repo_type="dataset", local_dir=RAW + "/screenspot", allow_patterns=["*.parquet"])
snapshot_download("allenai/pixmo-docs", repo_type="dataset", local_dir=RAW + "/pixmo-docs", allow_patterns=["*/validation-*"])

In [ ]:
if not os.path.isdir(D + "/coco/val2017"):
    !cd /content && curl -sSfLO http://images.cocodataset.org/zips/val2017.zip && mkdir -p $D/coco && unzip -q -o val2017.zip -d $D/coco && rm -f val2017.zip
!cd /content/laya && python research/vision/extract_eval_images.py --data $D --raw $RAW

missing = sorted(p for p in need if not os.path.exists(os.path.join(D, p)))
print("eval images:", len(need), "missing:", len(missing), missing[:10])
assert not missing, "some eval images are missing: send this output back"

## 3. Laya-V latency on the T4

In [ ]:
OUT = DRIVE_DIR + "/runs"
!cd /content/laya && python research/vision/bench_latency.py --ckpt $CK --items $D/items/stage_b.jsonl --image-root $D --usd-per-hour $USD_PER_HOUR --out "$OUT/latency_t4_real.json" 2>&1 | grep --line-buffered -v Warning | tee "$OUT/logs/latency.log"

## 4. Benchmark: Laya-V accuracy on the T4, then the VLMs on OpenRouter

The teacher runs first (its free preview ends around 29 Sep 2026). Laya-V's overall accuracy should come out
at about 58.7% on 4,872 rows, matching the published eval report.

In [ ]:
models = ("--models " + " ".join(MODELS)) if MODELS else ""
!cd /content/laya && python research/vision/bench_vlms.py --items $D/items/stage_b.jsonl --image-root $D --out "$OUT" --final $CK --audit-items $D/items/audit_items.jsonl --audit-verdicts $D/items/audit_verdicts_sonnet5.json --per-task $PER_TASK --max-usd $MAX_USD --workers 2 --rows 64 $models 2>&1 | grep --line-buffered -v Warning | tee "$OUT/logs/bench.log"

## 5. Summary and results zip

In [ ]:
import shutil
from google.colab import files

b = json.load(open(OUT + "/bench_vlms.json"))
lat = json.load(open(OUT + "/latency_t4_real.json"))
pct = lambda x: "  –  " if x is None else "%5.1f%%" % (100 * x)
print("%-28s %8s %15s %8s %7s %7s %9s %9s %7s" % ("system", "macro", "95% CI", "audit", "ECE", "ECEfit",
                                                   "p50 ms", "$/1k", "failed"))
for k, s in b["systems"].items():
    ci = s.get("macro_gold_ci95") or [None, None]
    if k == "laya_v":
        ms, usd = lat["mid_q1"]["p50_ms"], lat.get("usd_per_1k_decisions")
    else:
        ms, usd = (s.get("latency") or {}).get("p50_ms"), s["cost"]["usd_per_1k_decisions"]
    print("%-28s %8s %7s-%7s %8s %7.3f %7s %9s %9s %7d" % (
        k[:28], pct(s["macro_gold"]), pct(ci[0]), pct(ci[1]), pct(s.get("audit_acc")), s["ece_raw"],
        "%.3f" % s["ece_fitted"] if s.get("ece_fitted") is not None else "–",
        "%.0f" % ms if ms else "–", "%.4f" % usd if usd is not None else "–", s["failed_rows"]))
print("\nLaya-V all-rows accuracy on this T4: %.4f (published: 0.5870)" % b["systems"]["laya_v"]["metrics"]["all"]["acc"])
print("latency:", {k: round(v["p50_ms"]) for k, v in lat.items() if isinstance(v, dict)})
print("API spend: $%.2f" % sum(s.get("cost", {}).get("usd_eval", 0) for s in b["systems"].values()))

zip_path = shutil.make_archive(DRIVE_DIR + "/laya-v-bench-results", "zip", OUT)
print("wrote", zip_path)
files.download(zip_path)